# D182 - MySQL Indexing Strategies

An index is an auxiliary data structure that helps MySQL locate rows without examining every row in a table. Good indexes reduce reads, joins, sorting, and grouping work. They also consume storage and add work to `INSERT`, `UPDATE`, and `DELETE`.

This notebook introduces index fundamentals and then covers InnoDB **clustered indexes**, **secondary (non-clustered) indexes**, primary and unique keys, composite and covering indexes, prefix and functional indexes, descending and invisible indexes, full-text indexes, inspection commands, execution plans, and maintenance practices.

## Learning objectives

You will learn how to:

- explain why indexes improve reads and slow writes;
- distinguish an InnoDB clustered index from a secondary index;
- create primary, unique, ordinary, and composite indexes;
- inspect indexes with `DESC`, `SHOW CREATE TABLE`, `SHOW INDEX`, and `INFORMATION_SCHEMA`;
- recognize index access in `EXPLAIN`;
- apply the leftmost-prefix rule;
- identify covering-index plans;
- use prefix, functional, descending, invisible, and full-text indexes;
- detect redundant indexes and make evidence-based index decisions.

## 1. Connect to MySQL

The notebook creates a disposable `index_lab` database. Defaults are `127.0.0.1:3306`, username `root`, and password `root`.

In [ ]:
import os
import mysql.connector
from mysql.connector import Error

MYSQL_CONFIG = {
    'host': os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    'port': int(os.environ.get('MYSQL_PORT', '3306')),
    'user': os.environ.get('MYSQL_USERNAME', 'root'),
    'password': os.environ.get('MYSQL_PASSWORD', 'root'),
}

server = mysql.connector.connect(**MYSQL_CONFIG)
cursor = server.cursor()
cursor.execute('CREATE DATABASE IF NOT EXISTS index_lab')
cursor.close()
server.close()

connection = mysql.connector.connect(**MYSQL_CONFIG, database='index_lab')
print('Connected:', connection.is_connected())
print('MySQL version:', connection.get_server_info())

## 2. D14/D16-style query helpers

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print('No rows returned.')
        return
    text_rows = [['NULL' if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    print(' | '.join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(' | '.join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [item[0] for item in cursor.description]
            rows = cursor.fetchall()
            print_rows(columns, rows)
            return rows
        affected = cursor.rowcount
        connection.commit()
        print(f'Statement completed. Affected rows: {affected:,}')
        return affected
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## 3. Index fundamentals

Without a useful index, MySQL may perform a full table scan: read every row, test the predicate, and keep matches. A B-tree index stores ordered key values, allowing equality lookup, range scanning, ordered retrieval, and prefix matching.

Indexes can support:

- `WHERE` equality and range predicates;
- join keys;
- `ORDER BY` and some `GROUP BY` operations;
- uniqueness enforcement;
- covering queries that need only indexed values.

Indexes have costs:

- additional disk and buffer-pool space;
- extra writes and page maintenance;
- longer bulk loads and schema changes;
- more optimizer choices;
- operational work for creation, removal, and monitoring.

## 4. Common MySQL index categories

| Category | Main purpose |
|---|---|
| `PRIMARY KEY` | Unique, non-null row identity; clustered key in InnoDB |
| `UNIQUE` | Enforce uniqueness; nullable unique columns may allow multiple `NULL` values |
| Ordinary `INDEX` | Improve lookup without enforcing uniqueness |
| Composite index | Index two or more columns in a defined order |
| Prefix index | Index the first characters/bytes of a string |
| Functional index | Index a deterministic expression |
| `FULLTEXT` | Word-oriented text search |
| `SPATIAL` | Spatial values and spatial search |
| Hash index | Equality-oriented index in engines such as `MEMORY`; InnoDB user indexes are B-trees |

`KEY` and `INDEX` are synonyms in MySQL DDL.

# Part A - Primary, Unique, and Clustered Indexes

## 5. Create sample tables

The table declarations include a primary key, unique key, foreign-key-supporting index, and ordinary indexes. Re-running this cell resets the lab tables.

In [ ]:
execute_sql('DROP TABLE IF EXISTS orders_index_demo')
execute_sql('DROP TABLE IF EXISTS customers_index_demo')
execute_sql('DROP TABLE IF EXISTS articles_index_demo')

execute_sql("""
CREATE TABLE customers_index_demo (
    customer_id INT NOT NULL AUTO_INCREMENT,
    email VARCHAR(120) NOT NULL,
    full_name VARCHAR(100) NOT NULL,
    state_code CHAR(2) NOT NULL,
    signup_date DATE NOT NULL,
    loyalty_points INT NOT NULL DEFAULT 0,
    PRIMARY KEY (customer_id),
    CONSTRAINT uq_customer_email UNIQUE (email),
    INDEX idx_customer_state (state_code),
    INDEX idx_signup_date (signup_date)
) ENGINE=InnoDB
""")

execute_sql("""
CREATE TABLE orders_index_demo (
    order_id BIGINT NOT NULL,
    customer_id INT NOT NULL,
    order_date DATETIME NOT NULL,
    status VARCHAR(20) NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (order_id),
    INDEX idx_order_customer (customer_id),
    CONSTRAINT fk_order_customer FOREIGN KEY (customer_id)
        REFERENCES customers_index_demo(customer_id)
) ENGINE=InnoDB
""")

In [ ]:
customers = [
    ('asha@example.com', 'Asha Rao', 'KA', '2023-01-15', 900),
    ('bala@example.com', 'Bala Kumar', 'TN', '2023-04-10', 300),
    ('chitra@example.com', 'Chitra Nair', 'KL', '2024-02-20', 650),
    ('deepak@example.com', 'Deepak Shah', 'MH', '2024-06-05', 100),
    ('esha@example.com', 'Esha Iyer', 'KA', '2025-01-12', 1200),
    ('farhan@example.com', 'Farhan Ali', 'TN', '2025-03-08', 450),
]
orders = [
    (1001, 1, '2025-01-05 10:00:00', 'delivered', 1200.00),
    (1002, 1, '2025-02-12 11:30:00', 'delivered', 450.00),
    (1003, 2, '2025-02-15 09:10:00', 'cancelled', 250.00),
    (1004, 3, '2025-03-01 15:45:00', 'shipped', 800.00),
    (1005, 4, '2025-03-10 12:20:00', 'pending', 100.00),
    (1006, 5, '2025-04-02 16:05:00', 'delivered', 1500.00),
    (1007, 5, '2025-04-03 17:15:00', 'returned', 500.00),
    (1008, 6, '2025-04-08 08:40:00', 'delivered', 300.00),
]

cursor = connection.cursor()
try:
    cursor.executemany("""
        INSERT INTO customers_index_demo
        (email, full_name, state_code, signup_date, loyalty_points)
        VALUES (%s, %s, %s, %s, %s)
    """, customers)
    cursor.executemany("""
        INSERT INTO orders_index_demo
        (order_id, customer_id, order_date, status, amount)
        VALUES (%s, %s, %s, %s, %s)
    """, orders)
    connection.commit()
    print(f'Loaded {len(customers)} customers and {len(orders)} orders.')
except Error:
    connection.rollback()
    raise
finally:
    cursor.close()

## 6. Primary key

A primary key uniquely identifies every row. It is implicitly `NOT NULL`, and a table has at most one primary key, although that key may contain several columns. MySQL automatically creates an index to enforce it.

Good InnoDB primary keys are usually stable, narrow, and preferably increasing. Wide or frequently changing primary keys make every secondary index larger and costlier.

## 7. InnoDB clustered index

InnoDB stores table rows in the leaf pages of the clustered index. Therefore the clustered index is the table's physical B-tree organization, not a separate pointer structure.

InnoDB chooses the clustered key in this order:

1. the declared `PRIMARY KEY`;
2. otherwise, the first suitable `UNIQUE NOT NULL` index;
3. otherwise, an internal hidden 6-byte row ID.

A table has only one clustered index because rows can be physically organized only one way. MySQL does not display a `CLUSTERED` label in `SHOW INDEX`; this behavior comes from the InnoDB storage engine and key-selection rules.

In [ ]:
execute_sql('EXPLAIN SELECT * FROM customers_index_demo WHERE customer_id = 5')
execute_sql('SELECT * FROM customers_index_demo WHERE customer_id = 5')

For a constant primary-key lookup, `EXPLAIN` commonly reports access type `const`, key `PRIMARY`, and an estimate of one row.

## 8. Unique key

A unique index prevents duplicate non-null key combinations. Unlike a primary key, a unique index may contain nullable columns, and MySQL normally permits multiple `NULL` values because `NULL` is not equal to `NULL`. A table can have several unique indexes.

Use uniqueness to enforce a business rule, not merely as a performance hint. Here `email` must be unique.

In [ ]:
execute_sql("EXPLAIN SELECT customer_id, full_name FROM customers_index_demo WHERE email = 'esha@example.com'")

try:
    execute_sql("""
    INSERT INTO customers_index_demo
    (email, full_name, state_code, signup_date)
    VALUES ('esha@example.com', 'Duplicate Esha', 'KA', '2025-05-01')
    """)
except Error as exc:
    print('Expected unique-key error:', exc.msg)

# Part B - Secondary or Non-Clustered Indexes

## 9. How InnoDB secondary indexes work

An InnoDB secondary index stores the secondary key plus the row's primary-key value. It does not normally store a direct physical row address. When required columns are missing from the secondary index, InnoDB uses the stored primary key to look up the full row in the clustered index. This second lookup is often called a bookmark lookup or double lookup.

Consequences:

- a wide primary key makes every secondary index wider;
- selecting only indexed columns can avoid the clustered lookup;
- low-selectivity indexes may not be chosen when a large fraction of rows is needed.

In [ ]:
print('Secondary index, full row required')
execute_sql("EXPLAIN SELECT * FROM customers_index_demo WHERE state_code = 'KA'")

print('Potential covering query')
execute_sql("""
EXPLAIN SELECT customer_id, state_code
FROM customers_index_demo
WHERE state_code = 'KA'
""")

`Using index` in `Extra` means all required values can be read from the chosen index. It does not merely mean that an index is used; `key` tells you that.

# Part C - Inspecting Indexes

## 10. `DESCRIBE` / `DESC` table

`DESC table_name` shows columns and a compact `Key` indicator:

- `PRI`: column belongs to a primary key;
- `UNI`: column begins a unique index;
- `MUL`: column begins a nonunique index or can contain repeated indexed values.

`DESC` is convenient but incomplete. It does not reliably show full composite order, included key parts, visibility, cardinality, or all index properties.

In [ ]:
execute_sql('DESC customers_index_demo')

## 11. `SHOW CREATE TABLE`

This returns the authoritative DDL, including primary, unique, ordinary, and foreign-key definitions.

In [ ]:
execute_sql('SHOW CREATE TABLE customers_index_demo')

## 12. `SHOW INDEX`

`SHOW INDEX FROM table` is the standard detailed listing. Important fields include:

- `Non_unique`: 0 for primary/unique, 1 for nonunique;
- `Key_name`: index name;
- `Seq_in_index`: column position within a composite index;
- `Column_name` or `Expression`;
- `Collation`: `A` ascending, `D` descending;
- `Cardinality`: estimated distinct values;
- `Sub_part`: indexed prefix length;
- `Index_type`: normally `BTREE`, or `FULLTEXT`, `HASH`, and so on;
- `Visible`: whether the optimizer can normally choose it.

In [ ]:
execute_sql('SHOW INDEX FROM customers_index_demo')

## 13. `INFORMATION_SCHEMA.STATISTICS`

The metadata view is best for inventory queries across many schemas or tables. Cardinality is an estimate and can be refreshed with `ANALYZE TABLE`.

In [ ]:
execute_sql("""
SELECT index_name, non_unique, seq_in_index, column_name,
       collation, cardinality, sub_part, index_type, is_visible, expression
FROM information_schema.statistics
WHERE table_schema = DATABASE()
  AND table_name = 'customers_index_demo'
ORDER BY index_name, seq_in_index
""")

# Part D - Composite, Covering, and Specialized Indexes

## 14. Composite indexes and leftmost prefixes

A composite index stores columns in declared order. For `(status, order_date, customer_id)`, useful search prefixes begin on the left:

- `status`;
- `status, order_date`;
- `status, order_date, customer_id`.

A predicate only on `order_date` usually cannot seek efficiently through that index because the leading `status` value is unknown. One composite index is not equivalent to separate single-column indexes.

In [ ]:
execute_sql("""
CREATE INDEX idx_status_date_customer
ON orders_index_demo (status, order_date, customer_id)
""")
execute_sql('ANALYZE TABLE orders_index_demo')

print('Uses the leftmost prefix')
execute_sql("""
EXPLAIN SELECT order_id, customer_id, order_date
FROM orders_index_demo
WHERE status = 'delivered'
  AND order_date >= '2025-02-01'
ORDER BY order_date
""")

print('Missing the leftmost column')
execute_sql("""
EXPLAIN SELECT * FROM orders_index_demo
WHERE order_date >= '2025-02-01'
""")

### Equality, range, and column order

A common design places equality predicates first, followed by a range or ordering column. After MySQL uses a range on one key part, later parts often cannot further narrow the search interval, although they can still help filtering, ordering, or covering. Use `key_len`, JSON plans, and `EXPLAIN ANALYZE` to see actual behavior.

## 15. Covering index

A covering index contains every column required by a query. MySQL can answer it from index leaf pages without fetching the base row. Covering is a property of an index **for a particular query**, not a separate index type.

The composite order index contains `status`, `order_date`, and `customer_id`; InnoDB secondary leaves also include the primary key `order_id`.

In [ ]:
execute_sql("""
EXPLAIN SELECT order_id, customer_id, status, order_date
FROM orders_index_demo
WHERE status = 'delivered'
ORDER BY order_date
""")

## 16. Prefix indexes

Long strings can be indexed by a leading prefix to save space: `INDEX name (column(20))`. Prefix indexes can support leading-prefix predicates such as `LIKE 'asha%'`, but not leading wildcards such as `LIKE '%asha'`. They cannot cover the full original string value and may have weaker selectivity if the prefix is too short.

Choose prefix length from actual distinct-prefix measurements. Prefix lengths are subject to character set and maximum key-byte limits.

In [ ]:
execute_sql('CREATE INDEX idx_name_prefix ON customers_index_demo (full_name(5))')
execute_sql("EXPLAIN SELECT * FROM customers_index_demo WHERE full_name LIKE 'Asha%'")
execute_sql("EXPLAIN SELECT * FROM customers_index_demo WHERE full_name LIKE '%Rao'")

## 17. Functional indexes

A functional index stores the result of a deterministic expression. It can support queries that must apply that same expression to a column. Expression matching matters: a differently written expression may not use the index. Functional indexes are implemented through hidden generated columns and have restrictions on allowed expressions.

In [ ]:
execute_sql("""
CREATE INDEX idx_lower_email
ON customers_index_demo ((LOWER(email)))
""")
execute_sql("""
EXPLAIN SELECT customer_id, email
FROM customers_index_demo
WHERE LOWER(email) = 'esha@example.com'
""")
execute_sql('SHOW INDEX FROM customers_index_demo')

## 18. Descending indexes

MySQL 8 supports descending key parts. Mixed-direction indexes can avoid sorting for matching order patterns. The index order must match the query's leading filters and sort directions. MySQL can scan an index backward, so a simple all-ascending/all-descending reversal may not need two indexes; mixed directions are where explicit direction is especially useful.

In [ ]:
execute_sql("""
CREATE INDEX idx_customer_date_desc
ON orders_index_demo (customer_id ASC, order_date DESC)
""")
execute_sql("""
EXPLAIN SELECT order_id, customer_id, order_date
FROM orders_index_demo
WHERE customer_id = 5
ORDER BY order_date DESC
""")

## 19. Invisible indexes

An invisible index is maintained on writes but ignored by the optimizer by default. It helps test whether an index can be removed without immediately dropping it. Primary indexes cannot be invisible, and indexes required by constraints need special care.

Do not leave unused invisible indexes indefinitely; they still consume space and write cost.

In [ ]:
execute_sql('ALTER TABLE customers_index_demo ALTER INDEX idx_signup_date INVISIBLE')
execute_sql('SHOW INDEX FROM customers_index_demo')
execute_sql("EXPLAIN SELECT * FROM customers_index_demo WHERE signup_date >= '2025-01-01'")

# Restore visibility so later users see the original design.
execute_sql('ALTER TABLE customers_index_demo ALTER INDEX idx_signup_date VISIBLE')

## 20. Full-text index

`FULLTEXT` indexes support word-based natural-language and Boolean searches through `MATCH(...) AGAINST(...)`. They are different from B-tree indexes and are not replacements for exact equality, prefix, or numeric/date range searches. Stopwords, minimum token length, language, and parser behavior affect matches.

In [ ]:
execute_sql("""
CREATE TABLE articles_index_demo (
    article_id INT PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    body TEXT NOT NULL,
    FULLTEXT INDEX ft_title_body (title, body)
) ENGINE=InnoDB
""")
execute_sql("""
INSERT INTO articles_index_demo VALUES
(1, 'MySQL indexes', 'B-tree indexes accelerate equality and range searches'),
(2, 'Window functions', 'Window functions calculate rankings and running totals'),
(3, 'Database tuning', 'Execution plans help tune indexes and SQL queries')
""")
execute_sql("""
SELECT article_id, title,
       MATCH(title, body) AGAINST('indexes' IN NATURAL LANGUAGE MODE) AS relevance
FROM articles_index_demo
WHERE MATCH(title, body) AGAINST('indexes' IN NATURAL LANGUAGE MODE)
ORDER BY relevance DESC
""")

## 21. Spatial and hash indexes

### Spatial

A `SPATIAL INDEX` indexes spatial data types such as `POINT` or `POLYGON` for spatial predicates. Correct coordinate systems, SRIDs, validity, and MySQL spatial functions require dedicated treatment. It is not a B-tree replacement for latitude and longitude stored as unrelated numeric columns.

### Hash

InnoDB user-created indexes are B-trees. InnoDB may maintain an internal adaptive hash index automatically, but users do not define it per table. The `MEMORY` engine supports explicit `USING HASH` indexes, which are good for equality lookups but not ordered range scans. Engine selection must consider durability and operational requirements, not only index type.

# Part E - Index DDL and Lifecycle

## 22. Ways to create indexes

Indexes can be declared inside `CREATE TABLE`, added later with `CREATE INDEX`, or added through `ALTER TABLE`.

```sql
CREATE INDEX idx_name ON table_name (column_name);
CREATE UNIQUE INDEX uq_name ON table_name (column_name);
ALTER TABLE table_name ADD INDEX idx_name (column_name);
ALTER TABLE table_name ADD CONSTRAINT uq_name UNIQUE (column_name);
ALTER TABLE table_name ADD PRIMARY KEY (column_name);
```

Creating a unique or primary index validates existing data and fails if duplicates or prohibited nulls exist. Adding a primary key may rebuild the InnoDB table because it changes clustered organization.

## 23. Rename and drop an index

Use `ALTER TABLE ... RENAME INDEX old TO new`. Ordinary indexes can be removed with `DROP INDEX name ON table` or `ALTER TABLE ... DROP INDEX`. Drop a primary key with `ALTER TABLE ... DROP PRIMARY KEY`. Constraint-backed indexes and foreign keys require careful dependency handling.

In [ ]:
execute_sql("""
ALTER TABLE customers_index_demo
RENAME INDEX idx_name_prefix TO idx_customer_name_prefix
""")
execute_sql('SHOW INDEX FROM customers_index_demo')
execute_sql('DROP INDEX idx_customer_name_prefix ON customers_index_demo')

## 24. Statistics and maintenance

`ANALYZE TABLE` refreshes statistics used for cardinality and cost estimates. `CHECK TABLE` checks table/index consistency. `OPTIMIZE TABLE` can rebuild and analyze storage but may be expensive and is not routine medicine. InnoDB normally manages B-tree balance automatically; manual rebuilding should respond to evidence and operational planning.

In [ ]:
execute_sql('ANALYZE TABLE customers_index_demo, orders_index_demo')
execute_sql('CHECK TABLE customers_index_demo, orders_index_demo')
execute_sql("""
SELECT table_name, table_rows, data_length, index_length
FROM information_schema.tables
WHERE table_schema = DATABASE()
ORDER BY table_name
""")

# Part F - Index Design Best Practices

## 25. Evidence-based workflow

1. Start with a correct, important query and representative parameters.
2. Capture `EXPLAIN` and, when safe, `EXPLAIN ANALYZE`.
3. Identify filters, joins, ordering, grouping, selected columns, and actual row counts.
4. Design the narrowest useful index with deliberate column order.
5. Add one change at a time and refresh statistics.
6. Confirm the chosen key, access type, rows, loops, sorting, and actual timing.
7. Measure write overhead and index storage.
8. Look for overlapping or redundant indexes.
9. Monitor after deployment as data volume and distribution change.

## 26. Frequent mistakes

- Indexing every column without a workload.
- Using a wide, random, or mutable clustered primary key without considering secondary-index cost.
- Assuming a low-cardinality index must or must not be used; cost depends on the query and data.
- Ignoring composite column order and the leftmost-prefix rule.
- Duplicating an existing left prefix, such as keeping `(seller_id)` beside `(seller_id, date)` without a measured reason.
- Hiding indexed columns inside functions or implicit type conversions.
- Expecting `LIKE '%text'` to seek through a normal B-tree.
- Confusing `Using index` with merely choosing an index.
- Treating estimated cardinality as an exact count.
- Adding covering columns until indexes become excessively wide.
- Dropping an index required by a foreign key or uniqueness rule.
- Benchmarking only once on a warm or cold cache.

## 27. Practice tasks

1. Add a composite index for customer and amount, then inspect its column order.
2. Compare plans for `status` alone, `status + date`, and `date` alone.
3. Create a query covered by `idx_customer_date_desc`; then add `amount` to the select list and compare.
4. Measure distinct name prefixes of length 2, 3, 5, and 10 before choosing a prefix.
5. Make a nonconstraint index invisible and verify the plan change.
6. Find indexes whose leading columns overlap in `INFORMATION_SCHEMA.STATISTICS`.
7. Explain why a UUID primary key changes the cost of all InnoDB secondary indexes.

## 28. Close the connection

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed.')